# NLP feature engineering — heuristic keyword scores

A fast, fully interpretable NLP baseline before sentence embeddings. For
each `review` body we count curated flavour/aroma descriptor keywords and
turn them into numeric features:

- `kw_<feature>_count` — raw keyword hits in the review
- `kw_<feature>`       — hits per 100 words (density, length-normalised)

This is the **only** place keyword scoring lives. Output is saved keyed
by `wine_id` as `features_keywords.parquet`; the model notebook loads it and
joins on `wine_id` rather than recomputing.

In [1]:
import re
import numpy as np
import pandas as pd
import itables
from itables import show

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

SILVER_PATH   = r"..\..\.data\wine_reviews_silver.parquet"
KEYWORDS_PATH = r"..\..\features\features_keywords.parquet"

df = pd.read_parquet(SILVER_PATH)
print(f"Silver shape: {df.shape}")
print(f"reviews missing: {df['review'].isna().sum():,}")

Silver shape: (135192, 24)
reviews missing: 17


## Keyword dictionaries

Curated descriptor lists per flavour/aroma axis. Edit freely — the rest
of the notebook is driven entirely by this dict.

In [2]:
FEATURE_KEYWORDS = {
    "fruity": ["fruit", "fruity", "berry", "berries", "cherry", "cherries",
        "plum", "apple", "peach", "apricot", "citrus", "lemon", "lime",
        "orange", "grapefruit", "blackberry", "raspberry", "strawberry",
        "blueberry", "currant", "cassis", "melon", "pear", "fig", "tropical",
        "pineapple", "mango", "cranberry", "pomegranate"],
    "tannic": ["tannin", "tannins", "tannic", "grippy", "grip", "astringent",
        "firm", "chewy", "structured", "structure"],
    "acidic": ["acid", "acidity", "acidic", "crisp", "bright", "zesty", "zest",
        "tart", "fresh", "freshness", "lively", "vibrant", "racy", "tangy"],
    "oaky": ["oak", "oaky", "oaked", "vanilla", "cedar", "toast", "toasty",
        "smoky", "smoke", "barrel", "clove", "cinnamon", "spice", "spicy",
        "mocha", "espresso", "chocolate"],
    "sweet": ["sweet", "sweetness", "honey", "honeyed", "sugar", "sugary",
        "dessert", "candied", "jammy", "jam", "ripe", "luscious", "syrupy"],
    "body": ["full-bodied", "light-bodied", "medium-bodied", "rich", "heavy",
        "weighty", "concentrated", "dense", "powerful", "robust", "opulent",
        "lush", "plush"],
    "earthy": ["earth", "earthy", "mineral", "minerality", "leather",
        "leathery", "mushroom", "forest", "tobacco", "smoke", "flint", "stony"],
    "floral": ["floral", "flower", "rose", "violet", "blossom", "lavender",
        "jasmine", "honeysuckle", "perfumed"],
}

print(f"{len(FEATURE_KEYWORDS)} features, "
      f"{sum(len(v) for v in FEATURE_KEYWORDS.values())} keywords total")

8 features, 117 keywords total


## Compile patterns and score

One case-insensitive, word-boundaried alternation regex per feature.
`\\b` keeps hyphenated terms like `full-bodied` intact and stops
`oak` from matching `croak`.

In [ ]:
def build_patterns(keyword_dict):
    patterns = {}
    for feature, words in keyword_dict.items():
        alt = "|".join(re.escape(w) for w in sorted(words, key=len, reverse=True))
        patterns[feature] = re.compile(rf"\b(?:{alt})\b", re.IGNORECASE)
    return patterns


WORD_RE = re.compile(r"\b\w+\b")
PATTERNS = build_patterns(FEATURE_KEYWORDS)


def score_review(text):
    text = text or ""
    n_words = max(len(WORD_RE.findall(text)), 1)
    out = {}
    for feature, pat in PATTERNS.items():
        hits = len(pat.findall(text))
        out[f"kw_{feature}_count"] = hits
        out[f"kw_{feature}"] = round(100 * hits / n_words, 3)
    return out


reviews = df["review"].fillna("").astype(str)
kw_df = pd.DataFrame.from_records([score_review(t) for t in reviews], index=df.index)

count_cols   = [c for c in kw_df.columns if c.endswith("_count")]
density_cols = [c for c in kw_df.columns if not c.endswith("_count")]
print(f"Scored {len(kw_df):,} reviews -> {len(count_cols)} count + {len(density_cols)} density cols")
kw_df[density_cols].head()

# ~30s to score the full corpus; cache to parquet for reuse

Scored 135,192 reviews -> 8 count + 8 density cols


,kw_fruity,kw_tannic,kw_acidic,kw_oaky,kw_sweet,kw_body,kw_earthy,kw_floral
0,4.082,0.0,2.041,4.082,2.041,2.041,0.0,0.0
1,7.500,0.0,0.000,2.500,0.000,2.500,0.0,0.0
2,8.696,0.0,0.000,0.000,0.000,0.000,0.0,0.0
3,4.000,0.0,0.000,4.000,4.000,0.000,0.0,0.0
4,4.762,0.0,0.000,4.762,0.000,4.762,0.0,0.0


## Coverage and distribution

Share of reviews mentioning each axis at least once, plus density stats.

In [4]:
coverage = (kw_df[count_cols] > 0).mean().rename("pct_reviews_with_hit").mul(100).round(1)
avg_hits = kw_df[count_cols].mean().rename("avg_count").round(2)
summary = pd.concat([coverage, avg_hits], axis=1)
summary.index = summary.index.str.replace("kw_", "").str.replace("_count", "")
summary = summary.sort_values("pct_reviews_with_hit", ascending=False)
show(summary.reset_index().rename(columns={"index": "feature"}))

Loading ITables v2.7.3 from the internet... (need help?)


In [5]:
kw_df[density_cols].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
kw_fruity,135192.0,5.169,3.418,0.0,2.778,4.762,7.143,31.579
kw_tannic,135192.0,1.402,2.002,0.0,0.000,0.000,2.500,19.048
kw_acidic,135192.0,2.475,3.038,0.0,0.000,1.961,3.846,34.783
kw_oaky,135192.0,1.635,2.416,0.0,0.000,0.000,2.703,26.087
kw_sweet,135192.0,0.958,1.666,0.0,0.000,0.000,2.000,21.053
kw_body,135192.0,0.966,1.741,0.0,0.000,0.000,1.923,23.077
kw_earthy,135192.0,0.715,1.428,0.0,0.000,0.000,0.000,15.789
kw_floral,135192.0,0.572,1.307,0.0,0.000,0.000,0.000,18.182


## Spot-check

Sample a few reviews against their scores. Manually confirm the keyword
counts reflect what the text actually says.

In [6]:
sample_idx = df.sample(8, random_state=7).index
show(
    pd.concat([df.loc[sample_idx, ["name", "review"]], kw_df.loc[sample_idx, count_cols]], axis=1)
    .reset_index(drop=True),
    maxBytes="2MB",
)

Loading ITables v2.7.3 from the internet... (need help?)


## Sanity check vs. price & rating

Quick correlation of the density features against the target (`retail`)
and `rating` to confirm they carry (weak) signal.

In [7]:
signal = (
    pd.concat([kw_df[density_cols], df[["retail", "rating"]]], axis=1)
    .corr()[["retail", "rating"]]
    .drop(index=["retail", "rating"])
    .round(3)
    .sort_values("retail", ascending=False)
)
signal.index = signal.index.str.replace("kw_", "")
show(signal.reset_index().rename(columns={"index": "feature"}))

Loading ITables v2.7.3 from the internet... (need help?)


## Save keyword features

Persist `wine_id` + all `kw_*` columns. The model notebook joins these on
`wine_id` (NOT `slug`, which is not unique).

In [8]:
out = pd.concat([df[["wine_id"]], kw_df], axis=1)
out.to_parquet(KEYWORDS_PATH, index=False)
print(f"Saved {out.shape[0]:,} rows x {out.shape[1]} cols -> {KEYWORDS_PATH}")

Saved 135,192 rows x 17 cols -> ..\..\.data\features_keywords.parquet
